# Phase 5 Analysis — Guest Catalogue and Property Distribution

Implements Steps A–D from `03_design_spec.md` and `04_analysis_angle_structure.md`.
See `05_implementation_context.md` for data source decisions, join strategy, and key counts.

**Prerequisites:**
- `data/32_entity_deduplication/dedup_persons.csv`
- `data/32_entity_deduplication/dedup_cluster_members.csv`
- `data/31_entity_disambiguation/manual/reconciled_data_summary.csv`
- `data/31_entity_disambiguation/raw_import/episode_guests_normalized.csv`
- `data/31_entity_disambiguation/raw_import/episode_metadata_normalized.csv`
- `data/20_candidate_generation/wikidata/projections/archive/core_persons.json`
- `data/20_candidate_generation/wikidata/projections/archive/instances.csv`
- `data/00_setup/broadcasting_programs.csv`

**Outputs:** `data/50_analysis/all/` (combined) and `data/50_analysis/<show_id>/` (per-show)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

repo_root = Path.cwd()
if not (repo_root / "speakermining").exists():
    for parent in list(repo_root.parents):
        if (parent / "speakermining").exists():
            repo_root = parent
            break

os.chdir(repo_root)
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from process.io_guardrails import atomic_write_csv, atomic_write_text

OUTPUT_DIR  = Path("data/50_analysis")
ALL_DIR     = OUTPUT_DIR / "all"      # combined all-shows analysis
PERSONS_DIR = OUTPUT_DIR / "persons"  # GDPR-sensitive person data (isolatable)
REF_DIR     = OUTPUT_DIR / "reference" # non-human Wikidata reference data (publishable)
for _d in [OUTPUT_DIR, ALL_DIR, PERSONS_DIR, REF_DIR]:
    _d.mkdir(parents=True, exist_ok=True)
ARCH = Path("data/20_candidate_generation/wikidata/projections/archive")
print(f"repo_root: {repo_root}")
print(f"outputs:   {OUTPUT_DIR.resolve()}")
print(f"  all/       → combined analysis")
print(f"  persons/   → GDPR-sensitive person catalogue (exclude from public releases)")
print(f"  reference/ → non-human Wikidata data (publishable)")


## Setup
Load configuration, survey role values, define ROLE_MAP and helper functions.


In [ ]:
# --- In-scope broadcasting programs ---
broadcasting_programs = pd.read_csv(
    "data/00_setup/broadcasting_programs.csv", dtype=str
).fillna("")
IN_SCOPE_SHOW_IDS = {
    s.strip() for s in broadcasting_programs["fernsehserien_de_id"]
    if s.strip() and s.strip() != "NONE"
}
print(f"In-scope show IDs ({len(IN_SCOPE_SHOW_IDS)}): {sorted(IN_SCOPE_SHOW_IDS)}")

# --- Survey guest_role values ---
episode_guests_raw = pd.read_csv(
    "data/31_entity_disambiguation/raw_import/episode_guests_normalized.csv", dtype=str
).fillna("")
distinct_roles = sorted(episode_guests_raw["guest_role"].unique())
print(f"\nDistinct guest_role values: {distinct_roles}")

# --- Role mapping (data-driven from surveyed values; see 05_implementation_context.md §2) ---
ROLE_MAP = {
    "Gast":             "guest",
    "Kommentar":        "guest",
    "Kommentator":      "guest",
    "":                 "guest",      # empty = person listed on episode page, role unspecified
    "Moderation":       "moderator",
    "Produktionsauftrag": "staff",
    "Produktionsfirma": "staff",
    "Redaktion":        "staff",
    "Regie":            "staff",
    "Drehbuch":         "staff",
}
MODERATOR_QIDS = {"Q43773"}   # Markus Lanz — override for missing/ambiguous role data

# Roles ordered by analysis priority (lower = higher priority)
ROLE_PRIORITY = {"guest": 0, "moderator": 1, "staff": 2, "incidental": 3}

# --- Wikidata property extraction helpers ---
def _item_qid(stmt):
    try:
        return stmt["mainsnak"]["datavalue"]["value"]["id"]
    except (KeyError, TypeError):
        return ""

def _time_year(stmt):
    try:
        t = stmt["mainsnak"]["datavalue"]["value"]["time"]
        return t[1:5]
    except (KeyError, TypeError):
        return ""

def extract_props(claims):
    """Extract Phase 5 Wikidata properties from a claims dict."""
    gender_qid    = _item_qid(claims["P21"][0]) if "P21" in claims else ""
    occ_qids      = [_item_qid(s) for s in claims.get("P106", []) if _item_qid(s)]
    party_qids    = [_item_qid(s) for s in claims.get("P102", []) if _item_qid(s)]
    employer_qids = [_item_qid(s) for s in claims.get("P108", []) if _item_qid(s)]
    birthyear     = _time_year(claims["P569"][0]) if "P569" in claims else ""
    bp_qid        = _item_qid(claims["P19"][0]) if "P19" in claims else ""
    return gender_qid, occ_qids, party_qids, employer_qids, birthyear, bp_qid

print("\nSetup complete.")


In [ ]:
# --- Data loading ---
dedup_persons = pd.read_csv(
    "data/32_entity_deduplication/dedup_persons.csv", dtype=str
).fillna("")
cluster_members = pd.read_csv(
    "data/32_entity_deduplication/dedup_cluster_members.csv", dtype=str
).fillna("")
reconciled = pd.read_csv(
    "data/31_entity_disambiguation/manual/reconciled_data_summary.csv", dtype=str
).fillna("")
episode_meta = pd.read_csv(
    "data/31_entity_disambiguation/raw_import/episode_metadata_normalized.csv", dtype=str
).fillna("")

# Archive Wikidata data (archive/core_persons.json covers all 673 wikidata_id persons exactly)
core_persons = json.loads((ARCH / "core_persons.json").read_text(encoding="utf-8"))

# QID → label lookup (German preferred, fallback English)
instances = pd.read_csv(ARCH / "instances.csv", dtype=str).fillna("")
qid_label = {}
for _, row in instances.iterrows():
    label = row.get("labels_de", "") or row.get("labels_en", "") or row.get("label", "")
    if row["qid"] and label:
        qid_label[row["qid"]] = label
# Also index from core_persons labels
for qid, entity in core_persons.items():
    labels = entity.get("labels", {})
    label = labels.get("de", {}).get("value", "") or labels.get("en", {}).get("value", "")
    if label:
        qid_label[qid] = label

print(f"dedup_persons:         {len(dedup_persons):>7,} rows  ({(dedup_persons['wikidata_id'] != '').sum()} with wikidata_id)")
print(f"cluster_members:       {len(cluster_members):>7,} rows")
print(f"reconciled:            {len(reconciled):>7,} rows")
print(f"episode_guests_raw:    {len(episode_guests_raw):>7,} rows")
print(f"episode_meta:          {len(episode_meta):>7,} rows")
print(f"core_persons (arch):   {len(core_persons):>7,} entries")
print(f"qid_label lookup:      {len(qid_label):>7,} entries")


## Wikidata Property Fetch (`all_outlink_fetch`)

Fetches full entity claims (P21, P106, P102, P108, P569, P19, …) for all reconciled QIDs
not yet in the event log cache, using `entity_access.all_outlink_fetch` — the Phase 2
public interface for full property retrieval without inlinks expansion.

Idempotent: cache hits are skipped immediately. First run: ~15 min for ~4,500 missing QIDs
at 10 req/s. Subsequent runs: ~5–30 s (index priming, then all cache hits).
See TASK-A06 in `open-tasks.md`.


In [ ]:
try:
    from process.candidate_generation.wikidata.entity_access import (
        get_cached_entity_doc as _get_cached,
        all_outlink_fetch as _all_outlink_fetch,
        begin_request_context as _begin_ctx,
        end_request_context as _end_ctx,
    )
    _fetch_ok = True
except Exception as _e:
    print(f"WARNING: entity_access import failed: {_e}")
    print("Wikidata property coverage will be limited to archive/core_persons.json only.")
    _fetch_ok = False

needed_qids = sorted({
    qid for qid in reconciled["wikidata_id"].unique()
    if qid and qid != ""
})
print(f"Unique QIDs in reconciled: {len(needed_qids):,}")

if _fetch_ok:
    # Priming the cache index from entity_store.jsonl may take 10-30 s on first call
    print("Checking cache (index priming may take up to 30 s on first call)...")
    cached_before = sum(1 for q in needed_qids if _get_cached(q, repo_root) is not None)
    missing_qids  = [q for q in needed_qids if _get_cached(q, repo_root) is None]
    print(f"Already cached: {cached_before:,}  |  Missing: {len(missing_qids):,}")

    if missing_qids:
        est_min = len(missing_qids) / 10 / 60
        print(f"\nFetching {len(missing_qids):,} QIDs via all_outlink_fetch (~10 req/s, est. {est_min:.0f} min)...")
        fetched, failed = 0, 0
        # Initialize Phase 2 network guardrail — required before any network fetch
        _begin_ctx(
            budget_remaining=-1,           # unlimited: we need exactly len(missing_qids) calls
            query_delay_seconds=0.1,       # 10 req/s — within Wikidata API guidance
            progress_every_calls=500,
            context_label="phase5_property_fetch",
        )
        try:
            for i, qid in enumerate(missing_qids):
                try:
                    doc = _all_outlink_fetch(qid, repo_root)
                    if doc:
                        fetched += 1
                        # Supplement qid_label with labels from newly fetched docs
                        labels = doc.get("labels", {})
                        lbl = (labels.get("de", {}).get("value", "")
                               or labels.get("en", {}).get("value", ""))
                        if lbl and qid not in qid_label:
                            qid_label[qid] = lbl
                    else:
                        failed += 1
                except Exception as _exc:
                    failed += 1
                if (i + 1) % 500 == 0 or (i + 1) == len(missing_qids):
                    print(f"  [{i+1:,}/{len(missing_qids):,}] fetched={fetched:,} failed={failed:,}")
        finally:
            _n_requests = _end_ctx()
            print(f"  Network requests consumed by context: {_n_requests:,}")

        # Reset in-memory cache index so downstream cells see newly fetched docs
        try:
            from process.candidate_generation.wikidata.cache import reset_latest_cached_record_index
            reset_latest_cached_record_index(repo_root)
        except Exception:
            pass

        cached_after = sum(1 for q in needed_qids if _get_cached(q, repo_root) is not None)
        print(f"\nFetch complete.")
        print(f"  Coverage: {cached_after:,}/{len(needed_qids):,} ({100*cached_after/max(len(needed_qids),1):.1f}%)")
        print(f"  qid_label entries now: {len(qid_label):,}")
    else:
        print("All QIDs already cached — no network fetch needed.")
        print(f"  qid_label entries: {len(qid_label):,}")
else:
    print(f"Fetch skipped. qid_label entries from archive: {len(qid_label):,}")


## Audit: Unclassified Person Classification (TODO-040)

Some persons entered the candidate set via Wikidata discovery but were never linked to an actual episode.
They must appear in `person_catalogue_unclassified.csv` — not in the guest catalogue.
This cell checks the scale and composition of that group before Step A runs,
to verify no systematically misclassified guests are present.


In [ ]:
# Persons with no episode link in reconciled_data_summary
persons_no_episode = reconciled[
    (reconciled["entity_class"] == "person") &
    (reconciled["fernsehserien_de_id"] == "")
][["wikidata_id", "canonical_label", "match_strategy", "match_tier"]].drop_duplicates()

print(f"Persons with no episode link: {len(persons_no_episode):,}")
print(f"  With wikidata_id:    {(persons_no_episode['wikidata_id'] != '').sum():,}")
print(f"  Without wikidata_id: {(persons_no_episode['wikidata_id'] == '').sum():,}")
print(f"\nmatch_strategy breakdown:")
print(persons_no_episode["match_strategy"].value_counts().to_string())

print("\nRandom sample of 20 (verify these are genuinely unmatched — not missed guests):")
print(persons_no_episode.sample(min(20, len(persons_no_episode)), random_state=42).to_string(index=False))
print("\nThese persons will appear in person_catalogue_unclassified.csv after Step A.")
print("Review this list after Step A to confirm no systematically misclassified entries.")


## Step A: Build Person Catalogue

Builds `data/50_analysis/all/guest_catalogue.csv` (8,998 rows) from all canonical persons,
with role classification, appearance count, and Wikidata properties.

Join chain: `reconciled.alignment_unit_id` → `cluster_members` → `canonical_entity_id`
Role via: `reconciled.fernsehserien_de_id` (episode URL) + name → `episode_guests_normalized.guest_role`
In-scope filter: `episode_metadata.fernsehserien_de_id` (show ID) ∈ `IN_SCOPE_SHOW_IDS`


In [ ]:
# In-scope episode URLs
in_scope_episode_urls = set(
    episode_meta[episode_meta["fernsehserien_de_id"].isin(IN_SCOPE_SHOW_IDS)]["episode_url"]
)
print(f"In-scope episode URLs: {len(in_scope_episode_urls):,}")

# Join reconciled → cluster_members to get canonical_entity_id
cm_bridge = cluster_members[["alignment_unit_id", "canonical_entity_id"]].drop_duplicates("alignment_unit_id")
reconciled_ceid = reconciled.merge(cm_bridge, on="alignment_unit_id", how="left")
match_rate = reconciled_ceid["canonical_entity_id"].notna().mean()
print(f"Reconciled rows with canonical_entity_id: {match_rate:.1%}")

# Filter to in-scope episodes
reconciled_inscope = reconciled_ceid[
    reconciled_ceid["fernsehserien_de_id"].isin(in_scope_episode_urls)
].copy()
print(f"Reconciled rows in in-scope episodes: {len(reconciled_inscope):,}")

# Join with episode_guests_raw to get guest_role (key: episode_url + lowercase name)
eg_lookup = (
    episode_guests_raw[["episode_url", "guest_name", "guest_role"]]
    .copy()
    .assign(_name_lower=lambda d: d["guest_name"].str.strip().str.lower())
)
reconciled_inscope = reconciled_inscope.copy()
reconciled_inscope["_name_lower"] = reconciled_inscope["canonical_label"].str.strip().str.lower()

ri_with_role = reconciled_inscope.merge(
    eg_lookup.rename(columns={"episode_url": "fernsehserien_de_id", "guest_role": "raw_role"}),
    on=["fernsehserien_de_id", "_name_lower"],
    how="left"
).drop(columns=["_name_lower", "guest_name"], errors="ignore")

ri_with_role["raw_role"] = ri_with_role["raw_role"].fillna("")
ri_with_role["role"] = ri_with_role["raw_role"].map(ROLE_MAP).fillna("guest")
ri_with_role.loc[ri_with_role["wikidata_id"].isin(MODERATOR_QIDS), "role"] = "moderator"

print(f"\nRole distribution in in-scope appearances:")
print(ri_with_role["role"].value_counts().to_string())

# Appearance count per canonical entity
app_counts_s = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()]
    .groupby("canonical_entity_id")["fernsehserien_de_id"]
    .nunique()
    .rename("appearance_count")
    .reset_index()
)

# Dominant role per canonical entity (guest > moderator > staff > incidental)
dominant_role_s = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()]
    .groupby("canonical_entity_id")["role"]
    .agg(lambda roles: min(roles, key=lambda r: ROLE_PRIORITY.get(r, 9)))
    .rename("role")
    .reset_index()
)

# Derive best wikidata_id per canonical entity from reconciled_data_summary (authoritative)
# reconciled_ceid already contains reconciled.wikidata_id from the OpenRefine match
TIER_ORDER = {"high": 0, "medium": 1, "low": 2, "": 9}
best_qid_df = (
    reconciled_ceid[
        reconciled_ceid["canonical_entity_id"].notna() &
        (reconciled_ceid["wikidata_id"] != "")
    ][["canonical_entity_id", "wikidata_id", "match_tier"]]
    .assign(_rank=lambda d: d["match_tier"].map(TIER_ORDER).fillna(9).astype(int))
    .sort_values(["canonical_entity_id", "_rank"])
    .groupby("canonical_entity_id", as_index=False)
    .first()[["canonical_entity_id", "wikidata_id"]]
    .rename(columns={"wikidata_id": "reconciled_wikidata_id"})
)
print(f"Canonical entities with reconciled wikidata_id: {len(best_qid_df):,}")

# Build catalogue from dedup_persons (base: 8,998 rows)
catalogue = dedup_persons[[
    "canonical_entity_id", "wikidata_id", "canonical_label",
    "cluster_size", "cluster_strategy", "cluster_confidence"
]].copy()

catalogue = (
    catalogue
    .merge(dominant_role_s, on="canonical_entity_id", how="left")
    .merge(app_counts_s, on="canonical_entity_id", how="left")
    .merge(best_qid_df, on="canonical_entity_id", how="left")
)
catalogue["role"] = catalogue["role"].fillna("incidental")
# Override wikidata_id with the reconciled source (more complete than dedup_persons.wikidata_id)
catalogue["wikidata_id"] = (
    catalogue["reconciled_wikidata_id"]
    .where(catalogue["reconciled_wikidata_id"].notna() & (catalogue["reconciled_wikidata_id"] != ""),
           other=catalogue["wikidata_id"])
    .fillna("")
)
catalogue.drop(columns=["reconciled_wikidata_id"], inplace=True)
catalogue.loc[catalogue["wikidata_id"].isin(MODERATOR_QIDS), "role"] = "moderator"
catalogue["appearance_count"] = catalogue["appearance_count"].fillna(0).astype(int)
print(f"Persons with wikidata_id (reconciled override): {(catalogue['wikidata_id'] != '').sum():,}")

# Load entity_access for cache lookup — after all_outlink_fetch (cell 5b) this covers all reconciled QIDs
try:
    from process.candidate_generation.wikidata.entity_access import get_cached_entity_doc as _get_cached
    _entity_access_ok = True
except Exception:
    _entity_access_ok = False

# Extract Wikidata properties: archive first, entity_access cache fallback
coverage = {"archive": 0, "cache": 0, "empty": 0}
prop_records = []
for _, row in catalogue.iterrows():
    qid = row["wikidata_id"]
    entity = None
    if qid:
        entity = core_persons.get(qid)
        if entity:
            coverage["archive"] += 1
        elif _entity_access_ok:
            entity = _get_cached(qid, repo_root)
            if entity:
                coverage["cache"] += 1
                # Cache entity labels into qid_label for downstream use
                labels = entity.get("labels", {})
                lbl = labels.get("de", {}).get("value", "") or labels.get("en", {}).get("value", "")
                if lbl:
                    qid_label[qid] = lbl
            else:
                coverage["empty"] += 1
        else:
            coverage["empty"] += 1
    claims = entity.get("claims", {}) if entity else {}
    if claims:
        gender_qid, occ_qids, party_qids, employer_qids, birthyear, bp_qid = extract_props(claims)
        gender     = qid_label.get(gender_qid, gender_qid) if gender_qid else ""
        birthplace = qid_label.get(bp_qid, bp_qid) if bp_qid else ""
        occ_labels = [qid_label.get(q, q) for q in occ_qids]
        pty_labels = [qid_label.get(q, q) for q in party_qids]
        emp_labels = [qid_label.get(q, q) for q in employer_qids]
    else:
        gender_qid = gender = birthyear = bp_qid = birthplace = ""
        occ_qids = party_qids = employer_qids = []
        occ_labels = pty_labels = emp_labels = []
    prop_records.append({
        "canonical_entity_id": row["canonical_entity_id"],
        "gender": gender, "gender_qid": gender_qid,
        "birthyear": birthyear, "birthplace": birthplace, "birthplace_qid": bp_qid,
        "occupations":     "|".join(occ_labels), "occupation_qids":  "|".join(occ_qids),
        "party":           "|".join(pty_labels),  "party_qids":       "|".join(party_qids),
        "employer":        "|".join(emp_labels),  "employer_qids":    "|".join(employer_qids),
    })

props_df = pd.DataFrame(prop_records)
catalogue = catalogue.merge(props_df, on="canonical_entity_id", how="left")

CATALOGUE_COLS = [
    "canonical_entity_id", "wikidata_id", "canonical_label", "cluster_size",
    "cluster_strategy", "cluster_confidence", "role", "appearance_count",
    "gender", "gender_qid", "birthyear", "birthplace", "birthplace_qid",
    "occupations", "occupation_qids", "party", "party_qids", "employer", "employer_qids",
]
catalogue = catalogue[CATALOGUE_COLS]

# Write person catalogues to PERSONS_DIR (GDPR-sensitive — isolatable from public releases)
atomic_write_csv(PERSONS_DIR / "guest_catalogue.csv", catalogue)
unmatched    = catalogue[catalogue["wikidata_id"] == ""]
unclassified = catalogue[catalogue["appearance_count"] == 0]
atomic_write_csv(PERSONS_DIR / "guest_catalogue_unmatched.csv", unmatched)
atomic_write_csv(PERSONS_DIR / "person_catalogue_unclassified.csv", unclassified)

print(f"\npersons/guest_catalogue.csv:              {len(catalogue):,}")
print(f"persons/guest_catalogue_unmatched.csv:    {len(unmatched):,}  (no wikidata_id)")
print(f"persons/person_catalogue_unclassified.csv:{len(unclassified):,}  (appearance_count == 0)")
print(f"\nWikidata property coverage:")
print(f"  From archive core_persons.json: {coverage['archive']:,}")
print(f"  From entity_access cache:       {coverage['cache']:,}")
print(f"  No data (properties empty):     {coverage['empty']:,}")
print(f"\nRole distribution:")
print(catalogue["role"].value_counts().to_string())

# Sample the unclassified set for manual review
print(f"\nSample of unclassified persons (appearance_count == 0) for manual review:")
print(unclassified[["wikidata_id", "canonical_label", "cluster_strategy"]]
      .head(10).to_string(index=False))


## Step B: Occurrence Matrix

Builds the episode × guest matrix and derivative outputs.
Rows: canonical guests sorted by appearance_count desc.
Columns: in-scope episodes sorted by premiere_date asc.


In [ ]:
# Guest subset
guest_cat = catalogue[catalogue["role"] == "guest"].copy()
print(f"Guests in catalogue: {len(guest_cat):,}")

# Guest-episode pairs from ri_with_role
guest_ceids = set(guest_cat["canonical_entity_id"])
guest_pairs = ri_with_role[
    ri_with_role["canonical_entity_id"].isin(guest_ceids) &
    (ri_with_role["role"] == "guest")
][["canonical_entity_id", "fernsehserien_de_id"]].drop_duplicates()
print(f"Guest-episode pairs: {len(guest_pairs):,}")

# Episode sort order (by premiere_date asc)
ep_order = (
    episode_meta[episode_meta["episode_url"].isin(in_scope_episode_urls)]
    [["episode_url", "premiere_date", "fernsehserien_de_id", "program_name"]]
    .drop_duplicates("episode_url")
    .sort_values("premiere_date")
)
# Person sort order (appearance_count desc, then alpha)
person_order = (
    guest_cat[["canonical_entity_id", "canonical_label", "appearance_count"]]
    .sort_values(["appearance_count", "canonical_label"], ascending=[False, True])
)

# Pivot (1 = appeared, 0 = absent)
guest_pairs["_val"] = 1
matrix_num = guest_pairs.pivot_table(
    index="canonical_entity_id", columns="fernsehserien_de_id",
    values="_val", aggfunc="max", fill_value=0
)
ordered_persons  = [c for c in person_order["canonical_entity_id"] if c in matrix_num.index]
ordered_episodes = [e for e in ep_order["episode_url"] if e in matrix_num.columns]
matrix_num = matrix_num.reindex(index=ordered_persons, columns=ordered_episodes, fill_value=0)

# Output with 1/empty cells
matrix_out = matrix_num.copy().astype(object)
matrix_out[matrix_num == 0] = ""
ceid_to_label = person_order.set_index("canonical_entity_id")["canonical_label"]
matrix_out.insert(0, "canonical_label", ceid_to_label)
matrix_out = matrix_out.reset_index()
atomic_write_csv(ALL_DIR / "occurrence_matrix.csv", matrix_out)
print(f"all/occurrence_matrix.csv: {matrix_out.shape[0]} persons × {matrix_out.shape[1]-2} episodes")

# Per-show matrices — each in its own subdirectory
for show_id in sorted(IN_SCOPE_SHOW_IDS):
    show_eps = [e for e in ep_order[ep_order["fernsehserien_de_id"] == show_id]["episode_url"] if e in matrix_num.columns]
    if not show_eps:
        continue
    show_mat = matrix_num[show_eps]
    show_persons = show_mat[show_mat.sum(axis=1) > 0].index.tolist()
    show_out = matrix_out[matrix_out["canonical_entity_id"].isin(show_persons)][
        ["canonical_entity_id", "canonical_label"] + show_eps
    ].copy()
    show_dir = OUTPUT_DIR / show_id.replace("-", "_")
    show_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(show_dir / "occurrence_matrix.csv", show_out)
    print(f"  {show_dir.name}/occurrence_matrix.csv: {show_out.shape[0]} persons × {len(show_eps)} episodes")

# Co-occurrence matrix (top 200 guests)
top200 = ordered_persons[:200]
top_num = matrix_num.reindex(top200).fillna(0).astype(int)
co_occ = top_num.dot(top_num.T)
co_occ_out = co_occ.reset_index()
atomic_write_csv(ALL_DIR / "co_occurrence_matrix.csv", co_occ_out)
print(f"all/co_occurrence_matrix.csv: {co_occ_out.shape}")

# Completeness check (TODO-027)
recon_pairs = set(zip(
    reconciled_inscope["fernsehserien_de_id"],
    reconciled_inscope["canonical_label"].str.strip().str.lower()
))
eg_inscope = episode_guests_raw[episode_guests_raw["episode_url"].isin(in_scope_episode_urls)]
eg_pairs = set(zip(
    eg_inscope["episode_url"],
    eg_inscope["guest_name"].str.strip().str.lower()
))
print(f"\nCompleteness check (TODO-027):")
print(f"  In reconciled but not in episode_guests: {len(recon_pairs - eg_pairs):,}")
print(f"  In episode_guests but not in reconciled: {len(eg_pairs - recon_pairs):,}")

# Build episode_appearances for Steps C (one row per person-episode with properties)
episode_appearances = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()][
        ["canonical_entity_id", "fernsehserien_de_id", "role"]
    ]
    .merge(
        catalogue[["canonical_entity_id", "canonical_label", "wikidata_id", "gender",
                   "gender_qid", "birthyear", "occupations", "occupation_qids",
                   "party", "party_qids", "appearance_count"]],
        on="canonical_entity_id", how="left"
    )
    .merge(
        ep_order[["episode_url", "premiere_date", "fernsehserien_de_id", "program_name"]]
        .rename(columns={"fernsehserien_de_id": "show_id"}),
        left_on="fernsehserien_de_id", right_on="episode_url", how="left"
    )
)
episode_appearances["premiere_year"] = episode_appearances["premiere_date"].str[:4]
print(f"\nepisode_appearances DataFrame: {len(episode_appearances):,} rows")


## Step C1–C2: Gender Distribution

C1: Gender distribution across guest population (by person count and appearance count).
C2: Gender distribution over time (by year).


In [ ]:
guests = catalogue[catalogue["role"] == "guest"].copy()
print(f"Guest population: {len(guests):,} persons, {guests['appearance_count'].sum():,} total appearances")

# C1 — Gender distribution (F1)
c1 = (
    guests
    .groupby(guests["gender"].where(guests["gender"] != "", other="(unknown)"), dropna=False)
    .agg(person_count=("canonical_entity_id", "count"),
         appearance_count=("appearance_count", "sum"))
    .reset_index()
    .rename(columns={"gender": "value"})
    .sort_values("person_count", ascending=False)
)
total_p = c1["person_count"].sum()
total_a = c1["appearance_count"].sum()
c1["pct_by_person"]     = (c1["person_count"]     / total_p * 100).round(2)
c1["pct_by_appearance"] = (c1["appearance_count"] / total_a * 100).round(2)
atomic_write_csv(ALL_DIR / "gender_distribution.csv", c1)
print("\n=== C1: GENDER DISTRIBUTION ===")
print(c1.to_string(index=False))

# C1 per-show
c1_per_show = []
ep_guests = episode_appearances[episode_appearances["role"] == "guest"].copy()
for show_id in sorted(IN_SCOPE_SHOW_IDS):
    show_df = ep_guests[ep_guests["show_id"] == show_id]
    if show_df.empty:
        continue
    grp = (
        show_df
        .groupby(show_df["gender"].where(show_df["gender"] != "", other="(unknown)"), dropna=False)
        .agg(person_count=("canonical_entity_id", "nunique"),
             appearance_count=("canonical_entity_id", "count"))
        .reset_index()
        .rename(columns={"gender": "value"})
    )
    grp["show_id"] = show_id
    c1_per_show.append(grp)
if c1_per_show:
    c1_show_df = pd.concat(c1_per_show, ignore_index=True)
    atomic_write_csv(ALL_DIR / "gender_distribution_by_show.csv", c1_show_df)
    print(f"\nall/gender_distribution_by_show.csv: {len(c1_show_df)} rows")

# C2 — Gender over time (F2)
c2_rows = ep_guests[["canonical_entity_id", "premiere_year", "gender", "show_id"]].copy()
c2_rows["gender"] = c2_rows["gender"].where(c2_rows["gender"] != "", other="(unknown)")
c2 = (
    c2_rows
    .groupby(["premiere_year", "gender"], dropna=False)
    .agg(person_count=("canonical_entity_id", "nunique"),
         appearance_count=("canonical_entity_id", "count"))
    .reset_index()
    .sort_values(["premiere_year", "gender"])
)
atomic_write_csv(ALL_DIR / "gender_over_time.csv", c2)
print(f"\n=== C2: GENDER OVER TIME (first rows) ===")
print(c2.head(10).to_string(index=False))


## Step C3–C4: Occupation Distribution

C3: Occupation distribution with P279 subclustering (F1 + F5).
C4: Gender × occupation cross-tabulation (F3, top 20 occupations).


In [ ]:
# Load P279 class hierarchy for subclustering
crm_path = Path("data/20_candidate_generation/wikidata/projections/class_resolution_map.csv")
if crm_path.exists():
    class_res = pd.read_csv(crm_path, dtype=str).fillna("")
    # {child_qid -> top-level core_class_qid}
    qid_to_core = dict(zip(class_res["class_qid"], class_res["core_class_qid"]))
    print(f"class_resolution_map: {len(class_res):,} entries")
else:
    qid_to_core = {}
    print("WARNING: class_resolution_map.csv not found — using flat occupation list")

def map_to_core_class(occ_qid):
    return qid_to_core.get(occ_qid, occ_qid)

# C3 — Occupation distribution (F1, multi-value)
occ_df = guests.copy()
# Zip parallel columns before exploding — sequential explodes create a Cartesian product
occ_df["_occ_pairs"] = [
    list(zip(q.split("|"), l.split("|"))) if q and l else []
    for q, l in zip(occ_df["occupation_qids"].fillna(""), occ_df["occupations"].fillna(""))
]
occ_df = occ_df.explode("_occ_pairs")
occ_df = occ_df[occ_df["_occ_pairs"].notna()]
occ_df["occ_qid"] = occ_df["_occ_pairs"].apply(lambda x: x[0].strip() if isinstance(x, tuple) else "")
occ_df["occ"]     = occ_df["_occ_pairs"].apply(lambda x: x[1].strip() if isinstance(x, tuple) else "")
occ_df = occ_df[occ_df["occ"] != ""].drop(columns=["_occ_pairs"])

# Add core-class grouping
occ_df["core_class_qid"]   = occ_df["occ_qid"].map(map_to_core_class)
occ_df["core_class_label"] = occ_df["core_class_qid"].map(lambda q: qid_label.get(q, q) if q else "")

c3 = (
    occ_df
    .groupby(["occ", "occ_qid", "core_class_qid", "core_class_label"], dropna=False)
    .agg(person_count=("canonical_entity_id", "nunique"),
         appearance_count=("appearance_count", "sum"))
    .reset_index()
    .sort_values("person_count", ascending=False)
)
total_p = guests["canonical_entity_id"].nunique()
c3["pct_by_person"] = (c3["person_count"] / total_p * 100).round(2)
atomic_write_csv(ALL_DIR / "occupation_distribution.csv", c3)
print(f"=== C3: TOP 20 OCCUPATIONS ===")
print(c3.head(20)[["occ", "core_class_label", "person_count", "pct_by_person"]].to_string(index=False))

# C4 — Gender × occupation cross-tab (F3, top 20 occupations)
top20_occs = c3.head(20)["occ"].tolist()
c4_df = occ_df[occ_df["occ"].isin(top20_occs)].copy()
c4_df["gender"] = c4_df["gender"].where(c4_df["gender"] != "", other="(unknown)")
c4 = (
    c4_df
    .groupby(["occ", "gender"])["canonical_entity_id"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)
atomic_write_csv(ALL_DIR / "gender_by_occupation.csv", c4)
print(f"\n=== C4: GENDER × OCCUPATION (top 20 occupations) ===")
print(c4.to_string(index=False))

# F5 — Occupation hierarchy: Sunburst (requires plotly; see TASK-A02 for proper viz step)
try:
    import plotly.express as px
    sunburst_df = (
        c3[c3["core_class_label"] != ""]
        .groupby(["core_class_label", "occ"])
        .agg(person_count=("person_count", "sum"))
        .reset_index()
    )
    fig = px.sunburst(
        sunburst_df, path=["core_class_label", "occ"], values="person_count",
        title="Occupation Hierarchy — Guest Population (F5)",
    )
    fig.write_html(str(ALL_DIR / "occupation_sunburst.html"))
    fig.write_image(str(ALL_DIR / "occupation_sunburst.png"))
    print("\nall/occupation_sunburst.html / .png written")
    fig.show()
except Exception as e:
    print(f"\nSunburst skipped: {e}")


## Step C5–C7: Party Distribution

C5: Party affiliation distribution (F1).
C6: Gender × party cross-tabulation (F3, top 15 parties).
C7: Occupation × party cross-tabulation (F3, top 15 occ × top 10 parties).


In [ ]:
# C5 — Party distribution (F1, multi-value)
pty_df = guests.copy()
# Zip parallel columns before exploding — sequential explodes create a Cartesian product
pty_df["_pty_pairs"] = [
    list(zip(q.split("|"), l.split("|"))) if q and l else []
    for q, l in zip(pty_df["party_qids"].fillna(""), pty_df["party"].fillna(""))
]
pty_df = pty_df.explode("_pty_pairs")
pty_df = pty_df[pty_df["_pty_pairs"].notna()]
pty_df["pty_qid"] = pty_df["_pty_pairs"].apply(lambda x: x[0].strip() if isinstance(x, tuple) else "")
pty_df["pty"]     = pty_df["_pty_pairs"].apply(lambda x: x[1].strip() if isinstance(x, tuple) else "")
pty_df = pty_df[pty_df["pty"] != ""].drop(columns=["_pty_pairs"])

c5 = (
    pty_df
    .groupby(["pty", "pty_qid"], dropna=False)
    .agg(person_count=("canonical_entity_id", "nunique"),
         appearance_count=("appearance_count", "sum"))
    .reset_index()
    .rename(columns={"pty": "party", "pty_qid": "party_qid"})
    .sort_values("person_count", ascending=False)
)
total_p = guests["canonical_entity_id"].nunique()
c5["pct_by_person"] = (c5["person_count"] / total_p * 100).round(2)
atomic_write_csv(ALL_DIR / "party_distribution.csv", c5)
print("=== C5: TOP 15 PARTIES ===")
print(c5.head(15)[["party", "person_count", "pct_by_person"]].to_string(index=False))

# C6 — Gender × party (F3, top 15 parties)
top15_parties = c5.head(15)["party"].tolist()
pty_df["gender"] = pty_df["gender"].where(pty_df["gender"] != "", other="(unknown)")
c6 = (
    pty_df[pty_df["pty"].isin(top15_parties)]
    .groupby(["pty", "gender"])["canonical_entity_id"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={"pty": "party"})
)
atomic_write_csv(ALL_DIR / "gender_by_party.csv", c6)
print(f"\n=== C6: GENDER × PARTY ===")
print(c6.to_string(index=False))

# C7 — Occupation × party (F3, top 15 occ × top 10 parties)
top10_parties = c5.head(10)["party"].tolist()
top15_occs    = c3.head(15)["occ"].tolist() if "c3" in dir() else []
if top15_occs:
    occ_pty = (
        occ_df[occ_df["occ"].isin(top15_occs)]
        .merge(pty_df[["canonical_entity_id", "pty"]].drop_duplicates(),
               on="canonical_entity_id", how="left")
    )
    c7 = (
        occ_pty[occ_pty["pty"].isin(top10_parties)]
        .groupby(["occ", "pty"])["canonical_entity_id"]
        .nunique()
        .unstack(fill_value=0)
        .reset_index()
    )
    atomic_write_csv(ALL_DIR / "party_by_occupation.csv", c7)
    print(f"\n=== C7: OCCUPATION × PARTY ===")
    print(c7.to_string(index=False))


## Step C8: Age Distribution

Derived: `appearance_age = premiere_year − birth_year` (year-precision only).
Documented caveat: year-of-birth precision; episode recording may predate broadcast date.


In [ ]:
# C8 — Age distribution (F4, 10-year bins)
age_df = episode_appearances[
    (episode_appearances["role"] == "guest") &
    (episode_appearances["birthyear"] != "")
].copy()
age_df["birth_year_num"]    = pd.to_numeric(age_df["birthyear"], errors="coerce")
age_df["premiere_year_num"] = pd.to_numeric(age_df["premiere_year"], errors="coerce")
age_df = age_df.dropna(subset=["birth_year_num", "premiere_year_num"])
age_df["appearance_age"] = (age_df["premiere_year_num"] - age_df["birth_year_num"]).astype(int)
age_df = age_df[(age_df["appearance_age"] >= 0) & (age_df["appearance_age"] < 120)]

bins = list(range(0, 121, 10))
labels = [f"{b}-{b+9}" for b in bins[:-1]]
age_df["age_bin"] = pd.cut(age_df["appearance_age"], bins=bins, labels=labels, right=False)
c8 = (
    age_df.groupby("age_bin", observed=True)
    .agg(
        appearance_count=("canonical_entity_id", "count"),
        unique_persons=("canonical_entity_id", "nunique"),
        mean_age=("appearance_age", "mean"),
    )
    .reset_index()
)
c8["mean_age"] = c8["mean_age"].round(1)
atomic_write_csv(ALL_DIR / "age_distribution.csv", c8)
print("=== C8: AGE DISTRIBUTION ===")
print(c8.to_string(index=False))
print(f"\nCaveat: year-of-birth precision only; {age_df['appearance_age'].describe().round(1).to_dict()}")


## Step D1: Dataset Overview

Pipeline statistics: one row per phase/source with entity count and Wikidata coverage.


In [ ]:
# D1 — Dataset overview table
all_reconciled_persons = reconciled[reconciled["entity_class"] == "person"]
reconciled_with_qid    = all_reconciled_persons[all_reconciled_persons["wikidata_id"] != ""]

rows = [
    {"phase": "Phase 31", "source": "reconciled_data_summary",
     "entity_type": "person (appearance rows)",
     "total_count": len(all_reconciled_persons),
     "wikidata_matched_count": len(reconciled_with_qid),
     "coverage_pct": round(len(reconciled_with_qid) / len(all_reconciled_persons) * 100, 1)},
    {"phase": "Phase 32", "source": "dedup_persons",
     "entity_type": "canonical person",
     "total_count": len(dedup_persons),
     "wikidata_matched_count": (dedup_persons["wikidata_id"] != "").sum(),
     "coverage_pct": round((dedup_persons["wikidata_id"] != "").mean() * 100, 1)},
    {"phase": "Phase 5", "source": "guest_catalogue",
     "entity_type": "canonical person",
     "total_count": len(catalogue),
     "wikidata_matched_count": (catalogue["wikidata_id"] != "").sum(),
     "coverage_pct": round((catalogue["wikidata_id"] != "").mean() * 100, 1)},
    {"phase": "Phase 5", "source": "guest_catalogue (role=guest)",
     "entity_type": "canonical guest",
     "total_count": len(catalogue[catalogue["role"] == "guest"]),
     "wikidata_matched_count": (catalogue[catalogue["role"] == "guest"]["wikidata_id"] != "").sum(),
     "coverage_pct": None},
    {"phase": "Phase 5", "source": "person_catalogue_unclassified",
     "entity_type": "canonical person (no episode match)",
     "total_count": len(unclassified),
     "wikidata_matched_count": (unclassified["wikidata_id"] != "").sum(),
     "coverage_pct": None},
    {"phase": "Phase 5", "source": "episode_guests (in-scope)",
     "entity_type": "guest-episode pairs",
     "total_count": len(ri_with_role),
     "wikidata_matched_count": (ri_with_role["wikidata_id"] != "").sum(),
     "coverage_pct": round((ri_with_role["wikidata_id"] != "").mean() * 100, 1)},
    {"phase": "Phase 1 (raw)", "source": "episode_guests_normalized (all shows)",
     "entity_type": "guest-episode pairs",
     "total_count": len(episode_guests_raw),
     "wikidata_matched_count": None,
     "coverage_pct": None},
]

d1 = pd.DataFrame(rows)
atomic_write_csv(ALL_DIR / "dataset_overview.csv", d1)
print("=== D1: DATASET OVERVIEW ===")
print(d1.to_string(index=False))


## Dataset Preparation (TASK-A08)

Copies non-human Wikidata reference data from the Phase 2 archive into `data/50_analysis/reference/`
so that `data/50_analysis/` is a self-contained dataset — re-running the analysis requires no other folder.

**Directory structure:**
- `all/`       — combined all-shows analysis outputs (occurrence matrices, distributions, summary)
- `<show_id>/` — per-show occurrence matrices
- `persons/`   — GDPR-sensitive person data (names, birth years, properties) — exclude from public releases
- `reference/` — non-human Wikidata data (classes, hierarchy, labels) — publishable


In [ ]:
import shutil

# Non-human Wikidata reference files to include in the self-contained dataset
REF_FILES = [
    "classes.csv",
    "classes.parquet",
    "class_hierarchy.csv",
    "class_hierarchy.parquet",
    "instances.csv",
    "instances.parquet",
    "core_classes.csv",
    "core_classes.parquet",
    "core_roles.json",
    "core_organizations.json",
    "core_broadcasting_programs.json",
    "summary.json",
]

copied, skipped = [], []
for fname in REF_FILES:
    src = ARCH / fname
    dst = REF_DIR / fname
    if src.exists():
        shutil.copy2(src, dst)
        copied.append(fname)
    else:
        skipped.append(fname)

print(f"Reference data copied to {REF_DIR.resolve()}")
print(f"  Copied ({len(copied)}): {', '.join(copied)}")
if skipped:
    print(f"  Not found ({len(skipped)}): {', '.join(skipped)}")

# Report self-contained dataset structure
print(f"\n=== DATA/50_ANALYSIS DATASET STRUCTURE ===")
for subdir in sorted(OUTPUT_DIR.iterdir()):
    if subdir.is_dir():
        files = list(subdir.iterdir())
        print(f"  {subdir.name}/  ({len(files)} files)")
        for f in sorted(files)[:5]:
            print(f"    {f.name}")
        if len(files) > 5:
            print(f"    ... and {len(files)-5} more")


## Summary

Write `analysis_summary.json` with key statistics for downstream use.


In [ ]:
from collections import Counter

gender_dist = c1.set_index("value")[["person_count", "pct_by_person"]].to_dict("index")
top_occupations = c3.head(10)["occ"].tolist() if "c3" in dir() else []
top_parties     = c5.head(10)["party"].tolist() if "c5" in dir() else []
age_stats       = age_df["appearance_age"].describe().round(1).to_dict() if "age_df" in dir() else {}

summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "catalogue": {
        "total_canonical_persons": len(catalogue),
        "wikidata_matched": int((catalogue["wikidata_id"] != "").sum()),
        "role_distribution": catalogue["role"].value_counts().to_dict(),
    },
    "guests": {
        "total": int(len(guest_cat)),
        "with_wikidata": int((guest_cat["wikidata_id"] != "").sum()),
        "total_appearances": int(guest_cat["appearance_count"].sum()),
    },
    "gender_distribution_pct": {k: v["pct_by_person"] for k, v in gender_dist.items()},
    "top_occupations": top_occupations,
    "top_parties": top_parties,
    "age_stats": age_stats,
    "in_scope_shows": sorted(IN_SCOPE_SHOW_IDS),
    "output_files": [str(p.relative_to(OUTPUT_DIR)) for p in sorted(OUTPUT_DIR.rglob("*.csv"))],
}
atomic_write_text(ALL_DIR / "analysis_summary.json",
                  json.dumps(summary, indent=2, ensure_ascii=False))
print("all/analysis_summary.json written.")
print(json.dumps(summary, indent=2, ensure_ascii=False)[:2000])
